# F5-probability — Practice p25

**Type:** integrative · **Difficulty:** advanced · **Concepts:** conditional-probability, bayes-rule, hoeffding-inequality, sampling-simulation

| | Alert | No alert | Total |
| --- | ---: | ---: | ---: |
| Needs support | 54 | 6 | 60 |
| Does not need support | 94 | 846 | 940 |
| Total | 148 | 852 | 1000 |

**(a)** Compute `p_need_given_alert` directly. Separately compute `p_alert_given_need`, `p_need`, `p_alert_given_not_need`, then obtain `p_need_given_alert_bayes` through total probability. Explain denominator 148 versus 60 and the zero-denominator rule.

**(b)** Future independent audit scores satisfy $0\le X_i\le1$. With `N = 200`, `EPSILON = 0.10`, compute two-sided `raw_audit_bound` and capped `audit_envelope`. State assumptions, width, factor 2, and vacuity.

**(c)** Deploy only if posterior is at least 0.35 **and** envelope at most 0.05. Set `decision` to `"deploy"` or `"do_not_deploy"` and justify both conditions.

**(d)** With exactly `SEED = 20260804`, use the single vectorized count draw

`success_counts = rng.binomial(N, p_need_given_alert, size=N_TRIALS)`

for `N_TRIALS = 30_000`. Set `sample_means = success_counts / N` and record `empirical_audit_tail` from the inclusive count-space event `abs(success_counts - N * p_need_given_alert) >= N * EPSILON`. This simulation is evidence, not proof.

**Zero points:** loops/comprehensions, SciPy/pandas, hard-coded posterior/empirical frequency, a different RNG API/order, mean-space subtraction for tail membership, or one-sided Hoeffding constant.


In [ ]:
import numpy as np

N = 200
EPSILON = 0.10
SEED = 20260804
N_TRIALS = 30_000

need_and_alert = 54
need_total = 54 + 6
not_need_and_alert = 94
not_need_total = 94 + 846
alert_total = need_and_alert + not_need_and_alert
population = need_total + not_need_total

p_need_given_alert = need_and_alert / alert_total
p_alert_given_need = need_and_alert / need_total
p_need = need_total / population
p_alert_given_not_need = not_need_and_alert / not_need_total
evidence_probability = (p_alert_given_need * p_need +
                        p_alert_given_not_need * (1.0 - p_need))
p_need_given_alert_bayes = p_alert_given_need * p_need / evidence_probability
raw_audit_bound = float(2.0 * np.exp(-2.0 * N * EPSILON**2))
audit_envelope = min(1.0, raw_audit_bound)
decision = ("deploy" if p_need_given_alert >= 0.35 and audit_envelope <= 0.05
            else "do_not_deploy")
rng = np.random.default_rng(SEED)
success_counts = rng.binomial(N, p_need_given_alert, size=N_TRIALS)
sample_means = success_counts / N
empirical_audit_tail = (np.abs(success_counts - N * p_need_given_alert) >= N * EPSILON).mean()

**Your derivation and decision justification:**


### (a) Direct conditional and Bayes route

The direct posterior uses the Alert column:

$$
P(\text{Need}\mid\text{Alert})=\frac{54}{148}=\frac{27}{74}
\approx0.3648649.
$$

By contrast, (P(\text{Alert}\mid\text{Need})=54/60=0.9) uses the Needs-support row, so 148 and 60 are different conditioning denominators. With (P(\text{Need})=60/1000=0.06) and (P(\text{Alert}\mid\text{Not need})=94/940=0.1), total probability gives

$$
P(\text{Alert})=0.9(0.06)+0.1(0.94)=0.148,
$$

and Bayes returns (0.9(0.06)/0.148=27/74), matching the direct column calculation. If a conditioning total is zero, that conditional probability is undefined rather than zero, and a computational interface should raise an error.

### (b) Hoeffding bound

The future audit variables are assumed independent and each is bounded in ([0,1]), so every interval width is 1. The two-sided factor 2 gives

$$
2\exp(-2N\epsilon^2)=2e^{-4}
\approx0.0366312778.
$$

The capped envelope is the same value because the raw bound is below 1; it is informative rather than vacuous.

### (c) Decision

The posterior (27/74\approx0.3648649) meets the `0.35` threshold, and the envelope (0.0366312778) meets the `0.05` threshold. Both required conditions hold, so `decision = "deploy"`.

### (d) Simulation

The exact pinned vectorized count draw gives `empirical_audit_tail = 0.0031666666666666666` for the inclusive count-space event. This empirical frequency is reproducible evidence from 30,000 trials, not a proof or replacement for the Hoeffding guarantee.

In [ ]:
# Immutable semantic checks derive targets from the table, theorem, and pinned RNG API.
_need_and_alert = 54
_need_total = 54 + 6
_not_need_and_alert = 94
_not_need_total = 94 + 846
_population = _need_total + _not_need_total
_expected_direct = _need_and_alert / (_need_and_alert + _not_need_and_alert)
_expected_alert_given_need = _need_and_alert / _need_total
_expected_need = _need_total / _population
_expected_alert_given_not_need = _not_need_and_alert / _not_need_total
_expected_evidence = _expected_alert_given_need * _expected_need + _expected_alert_given_not_need * (1.0 - _expected_need)
_expected_bayes = _expected_alert_given_need * _expected_need / _expected_evidence
_expected_raw = 2.0 * np.exp(-2.0 * N * EPSILON**2)
_expected_envelope = min(1.0, _expected_raw)
_expected_decision = "deploy" if _expected_direct >= 0.35 and _expected_envelope <= 0.05 else "do_not_deploy"
_check_rng = np.random.default_rng(SEED)
_expected_counts = _check_rng.binomial(N, _expected_direct, size=N_TRIALS)
_expected_means = _expected_counts / N
_expected_empirical = (np.abs(_expected_counts - N * _expected_direct) >= N * EPSILON).mean()
assert np.isclose(p_need_given_alert, _expected_direct, atol=1e-12, rtol=0.0)
assert np.isclose(p_alert_given_need, _expected_alert_given_need, atol=1e-12, rtol=0.0)
assert np.isclose(p_need, _expected_need, atol=1e-12, rtol=0.0)
assert np.isclose(p_alert_given_not_need, _expected_alert_given_not_need, atol=1e-12, rtol=0.0)
assert np.isclose(p_need_given_alert_bayes, _expected_bayes, atol=1e-12, rtol=0.0)
assert np.isclose(p_need_given_alert, p_need_given_alert_bayes, atol=1e-12, rtol=0.0)
assert np.isclose(raw_audit_bound, _expected_raw, atol=1e-15, rtol=0.0)
assert np.isclose(audit_envelope, _expected_envelope, atol=1e-15, rtol=0.0)
assert decision == _expected_decision
assert isinstance(success_counts, np.ndarray) and success_counts.shape == (N_TRIALS,)
assert np.issubdtype(success_counts.dtype, np.integer)
assert np.all((0 <= success_counts) & (success_counts <= N))
assert np.array_equal(success_counts, _expected_counts)
assert int(success_counts.sum()) == int(_expected_counts.sum())
assert isinstance(sample_means, np.ndarray) and sample_means.shape == (N_TRIALS,)
assert np.issubdtype(sample_means.dtype, np.floating)
assert np.all((0.0 <= sample_means) & (sample_means <= 1.0))
assert np.allclose(sample_means, _expected_means, atol=0.0, rtol=0.0)
assert np.isclose(empirical_audit_tail, _expected_empirical, atol=1e-15, rtol=0.0)
assert 0.0 <= empirical_audit_tail <= audit_envelope <= 1.0
print("p25 checks passed")


### Answer check

The immutable assertions independently derive the direct and Bayes posteriors, two-sided Hoeffding bound, cap, and two-condition decision, then rebuild the exact pinned binomial count stream and compare integer counts, floating means, and the inclusive empirical tail with explicit tolerances. They also confirm count bounds, array shapes and dtypes, exact draw equality, and that the empirical frequency does not exceed the stated envelope.